# Kapitel 17

In [1]:
import math, random
from typing import Callable
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
def sigmoid(x): return expit(x) # = 1 / (1 + math.exp(-x)) aber robuster 
def distance(x,y): # Quadrat des Abstands zwischen zwei Vektoren
    return sum([(x[i]-y[i])**2 for i in range(len(x))])
sigmoid([1,2,3])

array([0.73105858, 0.88079708, 0.95257413])

In [2]:
def TDatenMonoton(n:int)->list:
    return [TDatumMonoton() for i in range(n)]
def TDatumMonoton():
    a=random.random(); b=random.random(); c=random.random()
    if a<=b and b<= c: 
        return [[a,b,c],[1,0]]
    if a>=b and b>= c: 
        return [[a,b,c],[0,1]]
    return  [[a,b,c],[0.5,0.5]] # Unschlüssig
TDatenMonoton(10)

[[[0.24961836745343102, 0.6375943838108182, 0.4602456583871556], [0.5, 0.5]],
 [[0.6799642648114887, 0.02413353171722743, 0.515271194983608], [0.5, 0.5]],
 [[0.9887941624009647, 0.31171001053237957, 0.6976084278859386], [0.5, 0.5]],
 [[0.8959291849719587, 0.2988945205048861, 0.5608345402138397], [0.5, 0.5]],
 [[0.16096508542463672, 0.035230954909976964, 0.3412346562483298], [0.5, 0.5]],
 [[0.8306935632733742, 0.9732854088129274, 0.09249638894684442], [0.5, 0.5]],
 [[0.2640624199349748, 0.5954478927421144, 0.7595402287139991], [1, 0]],
 [[0.6411223772061198, 0.4458514983387979, 0.4925549401507937], [0.5, 0.5]],
 [[0.3876696845672768, 0.8943367966689589, 0.11931288359516024], [0.5, 0.5]],
 [[0.6532028512114756, 0.22573505134539584, 0.46694006689098255], [0.5, 0.5]]]

In [3]:
# Erzeugung von je zwei Matrizen und Vektoren, zunächst mit Nullen gefüllt
W1=np.zeros((4, 3)) # 4x3 Matrix für die Gewichte von R^3->R^4 
W2=np.zeros((2, 4)) # 2x4 Matrix 
B1=np.zeros((4))
B2=np.zeros((2))
def M2V(W1,W2,B1,B2): # verwandelt die Vektoren und Matrizen in einen einzigen großen Vektor
    return np.hstack((W1.flatten(), W2.flatten(), B1.flatten(), B2.flatten()))
def V2M(x): # Umkehroperation zu M2V
    lenW1=np.prod(np.shape(W1))
    lenW2=np.prod(np.shape(W2))
    lenB1=np.prod(np.shape(B1))
    lenB2=np.prod(np.shape(B2))
    w1 = x[:lenW1].reshape(np.shape(W1))
    w2 = x[lenW1:lenW1+lenW2].reshape(np.shape(W2))
    b1 = x[lenW1+lenW2:lenW1+lenW2+lenB1].reshape(np.shape(B1))
    b2 = x[lenW1+lenW2+lenB1:].reshape(np.shape(B2))
    return [w1,w2,b1,b2]
x0 = M2V(W1, W2, B1, B2) # Die Matrizen als ein großer Vektor
[V2M(x0)[0]==W1, V2M(x0)[1]==W2, V2M(x0)[2]==B1, V2M(x0)[3]==B2] # Rückkonvertierung liefert wieder die Matrizen

[array([[ True,  True,  True],
        [ True,  True,  True],
        [ True,  True,  True],
        [ True,  True,  True]]),
 array([[ True,  True,  True,  True],
        [ True,  True,  True,  True]]),
 array([ True,  True,  True,  True]),
 array([ True,  True])]

In [4]:
def net(X,w1,w2,b1,b2): 
    return sigmoid(w2@sigmoid(w1@X+b1)+b2)
net([1,2,1],W1,W2,B1,B2)

array([0.5, 0.5])

In [5]:
# Das Folgende ist die zu minimierende Zielfunktion des Lernens: 
trainingdata=TDatenMonoton(500)
def F(x):
    [w1,w2,b1,b2]=V2M(x)
    return sum([distance(d[1],net(d[0],w1,w2,b1,b2)) for d in trainingdata])
opt=minimize(F,M2V(W1,W2,B1,B2))
[W1opt,W2opt,B1opt,B2opt]=V2M(opt.x)
# das Folgende ist das trainierte Netz
def netopt(X): 
    return [float(x) for x in list(net(X,W1opt,W2opt,B1opt,B2opt))]
def testit(x):
    print("net(",x,")=",netopt(x))
testit([-0.1,0.5,0.9])
testit([-0.1,0.8,1.2])
testit([-0.1,1.5,0.3])
testit([2.1,1.5,1.3])
testit([0.8,0.7,0.6])

net( [-0.1, 0.5, 0.9] )= [1.0, 2.221862663840962e-68]
net( [-0.1, 0.8, 1.2] )= [1.0, 2.221862663840962e-68]
net( [-0.1, 1.5, 0.3] )= [0.5028227529796232, 0.49720384519294825]
net( [2.1, 1.5, 1.3] )= [3.1424705981931877e-49, 1.0]
net( [0.8, 0.7, 0.6] )= [3.1207028228076624e-49, 1.0]


In [6]:
netopt(np.array([-0.1,1.5,0.3]))

[0.5028227529796232, 0.49720384519294825]

In [7]:
netopt(np.array([2.1,1.5,0.3]))

[3.1424705981931877e-49, 1.0]

In [8]:
def TDatenMinimum(n:int)->list:
    return [TDatumMinimum() for i in range(n)]
def TDatumMinimum():
    a=random.random(); b=random.random(); c=random.random()
    if a>=b and b<= c: 
        return [[a,b,c],[1,0]]
    return [[a,b,c],[0,1]]
trainingdata=TDatenMinimum(100)
def F(x):
    [w1,w2,b1,b2]=V2M(x)
    return sum([distance(d[1],net(d[0],w1,w2,b1,b2)) for d in trainingdata])
opt=minimize(F,M2V(W1,W2,B1,B2))
[W1opt,W2opt,B1opt,B2opt]=V2M(opt.x)
testit([0.5,0.6,1])
testit([-0.1,1.5,0.3])
testit([2.1,1.5,1.8])

net( [0.5, 0.6, 1] )= [1.1880047741479922e-119, 1.0]
net( [-0.1, 1.5, 0.3] )= [5.963972054657028e-259, 1.0]
net( [2.1, 1.5, 1.8] )= [0.9999999999999998, 1.1366029476372787e-11]


In [9]:
# Das ganze nochmal mit pytorch
# ggf. %pip install torch 

import torch
from torch import nn
from torch.optim import LBFGS # oder Adam

trainingdata = TDatenMonoton(100)
X = torch.tensor([d[0] for d in trainingdata], dtype=torch.float32)  
Y = torch.tensor([d[1] for d in trainingdata], dtype=torch.float32) 

# Netz: 3 -> 4 -> 2, Sigmoid
class MonoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 4, bias=True)
        self.fc2 = nn.Linear(4, 2, bias=True)
        self.sig = nn.Sigmoid()
        for p in self.parameters(): # Startwerte
            nn.init.constant_(p, 0.0)
    def forward(self, x): # Berechnung des Netzes
        h = self.sig(self.fc1(x)) 
        return self.sig(self.fc2(h))
net = MonoNet() # Das Netz

criterion = nn.MSELoss()
optimizer = LBFGS(net.parameters(), lr=0.8, 
                  max_iter=100, history_size=10, line_search_fn="strong_wolfe")

def closure():
    optimizer.zero_grad()
    yhat = net(X)
    loss = criterion(yhat, Y)
    loss.backward() # Backpropagation
    return loss

optimizer.step(closure)
# „trainiertes Netz“ wie oben
def netopt(x_list):
    x = torch.tensor(x_list, dtype=torch.float32).unsqueeze(0)  # (1,3)
    with torch.no_grad():
        y = net(x).squeeze(0).tolist()
    return [float(v) for v in y]

testit([-0.1, 1.5, 3.0])   # steigend -> nahe [1,0]
testit([-0.1, 1.5, 0.3])   # unsortiert -> nahe [0.5,0.5]
testit([ 2.1, 1.5, 1.3])   # fallend -> nahe [0,1]

net( [-0.1, 1.5, 3.0] )= [0.9990467429161072, 0.0009017311385832727]
net( [-0.1, 1.5, 0.3] )= [0.6823331713676453, 0.3162446916103363]
net( [2.1, 1.5, 1.3] )= [0.07085662335157394, 0.9304397106170654]
